# System Exploration (Security)

Parsing the data and understanding it (System attribute)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from anomaly_detection.etl.load import load_records, load_system_df

plt.style.use('ggplot')

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

output_path = project_folder / "data/system_exploration/security"

evtx_path = project_folder / "data/raw/93_securitelog.evtx"

evtx_path

In [ ]:
records = load_records(evtx_path)

len(records)

In [ ]:
with open(project_folder / "data/processed/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records[0],
        indent=4
    ))

records[0]

In [ ]:
system_df = load_system_df(evtx_path)

system_df.head()

In [ ]:
system_df.columns

# EventID

In [ ]:
events_df = pd.DataFrame(system_df['EventID'].value_counts())

total_events = events_df['count'].sum()

events_df['percentage'] = (events_df['count'] / total_events) * 100

events_df

In [ ]:
keep_list = events_df[events_df['count'] > 10]

keep_list.index

In [ ]:
# Add the descriptions to the event IDs
event_descriptions = {
    "4624": "An account was successfully logged on",
    "4672": "Special privileges assigned to new logon",
    "4634": "An account was logged off",
    "4648": "A logon was attempted using explicit credentials",
    "4776": "The computer attempted to validate the credentials for an account",
    "4799": "A security-enabled local group membership was enumerated",
    "4702": "A scheduled task was updated",
    "5379": "Credential Manager credentials were read",
    "4662": "An operation was performed on an object",
    "4697": "A service was installed in the system",
    "4798": "A user's local group membership was enumerated",
    "4611": "A trusted logon process has been registered with the Local Security Authority",
    "5058": "Key file operation",
    "5061": "Cryptographic operation",
    "5059": "Key migration operation",
    "4699": "A scheduled task was deleted",
    "4698": "A scheduled task was created",
}

desc_series = pd.Series(event_descriptions, name='description')
desc_series.index = desc_series.index.astype(events_df.index.dtype)  # match dtype (str vs int)

events_df = events_df.join(desc_series)

events_df.to_csv(output_path / "events.txt")

events_df

In [ ]:
unique_event_ids = [4611, 5058, 5061, 5059, 4699, 4698]

unique_events = [record for record in records if record['System']['EventID'] in unique_event_ids]

with open(output_path / "unique_event_records.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        unique_events,
        indent=4
    ))

unique_events

# Correlation

In [ ]:
correlation_activity_df = pd.DataFrame(system_df['Correlation_ActivityID'].value_counts())

correlation_activity_df

In [ ]:
main_corr_id = system_df[system_df["Correlation_ActivityID"] == "F1DAB952-E970-0004-5DB9-DAF170E9DC01"]["EventID"].value_counts()
other_corr_id = system_df[system_df["Correlation_ActivityID"] != "F1DAB952-E970-0004-5DB9-DAF170E9DC01"]["EventID"].value_counts()

corr_comparison_df = pd.concat(
    [main_corr_id, other_corr_id],
    axis=1,
    keys=['MainCorrelationActivityID', 'OtherCorrelationActivityID']
)

corr_comparison_df = corr_comparison_df.fillna(0)

corr_comparison_df

# Execution

In [ ]:
execution_df = pd.DataFrame(system_df[['Execution_ProcessID', 'Execution_ThreadID']].value_counts())

execution_df

In [ ]:
eventid_to_threadid_df = pd.crosstab(system_df["Execution_ThreadID"], system_df["EventID"])

eventid_to_threadid_df

# Others

In [ ]:
provider_df = pd.DataFrame(system_df[['Provider_Name', 'Provider_Guid']].value_counts())

provider_df

In [ ]:
system_df.astype(str).nunique()

In [ ]:
constants_df = pd.DataFrame(system_df[['Level', 'Opcode', 'Keywords', 'Channel', 'Computer', 'Execution_ProcessID']].value_counts())

constants_df.to_csv(output_path / "constants.csv")

constants_df

In [ ]:
eventid_to_task_df = pd.crosstab(system_df["Task"], system_df["EventID"])

eventid_to_task_df

In [ ]:
version_df = pd.DataFrame(system_df['Version'].value_counts())

version_df

In [ ]:
eventid_to_version_df = pd.crosstab(system_df["Version"], system_df["EventID"])

eventid_to_version_df